In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
dataset=pd.read_csv("CKD.csv")
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2.000000,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,0,0,0,0,0,0,1,1,0,1
1,3.000000,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,1,0,0,0,0,0,1,0,0,1
2,4.000000,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,1,0,0,0,0,0,1,0,0,1
3,5.000000,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,1,0,0,0,0,0,1,0,1,1
4,5.000000,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,1,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,0.0,0.0,219.000000,36.000000,1.300000,139.000000,3.700000,12.500000,...,1,0,0,0,0,0,1,0,0,1
395,51.492308,70.000000,0.0,2.0,220.000000,68.000000,2.800000,137.528754,4.627244,8.700000,...,1,0,0,1,1,0,1,0,1,1
396,51.492308,70.000000,3.0,0.0,110.000000,115.000000,6.000000,134.000000,2.700000,9.100000,...,1,0,0,1,1,0,0,0,0,1
397,51.492308,90.000000,0.0,0.0,207.000000,80.000000,6.800000,142.000000,5.500000,8.500000,...,1,0,0,1,1,0,1,0,1,1


In [3]:
dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [4]:
indep=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
dep=dataset[['classification_yes']]


In [5]:
from sklearn.model_selection import train_test_split
(x_train,x_test,y_train,y_test)=train_test_split(indep,dep,test_size=0.30,random_state=0)

In [6]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
param_grid={'solver':['lbfgs','liblinear','newton_cg','saga'],'penalty':['l1','l2']}
logistic_grid=GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
logistic_grid.fit(x_train,y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
15 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Anaconda3\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1193, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual

GridSearchCV(estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'penalty': ['l1', 'l2'],
                         'solver': ['lbfgs', 'liblinear', 'newton_cg', 'saga']},
             scoring='f1_weighted', verbose=3)

In [8]:

y_pred=logistic_grid.predict(x_test)
from sklearn.metrics import confusion_matrix  
cm=confusion_matrix(y_test,y_pred)
from sklearn.metrics import classification_report
clf_report=classification_report(y_test,y_pred)
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,y_pred,average='weighted')
print("The f1_macro value for best parameter{}:".format(logistic_grid.best_params_),f1_macro)
print("The confusion Matrix is:\n",cm)
print("The Cassification report is:\n",clf_report)


The f1_macro value for best parameter{'penalty': 'l2', 'solver': 'lbfgs'}: 0.9916844900066377
The confusion Matrix is:
 [[45  0]
 [ 1 74]]
The Cassification report is:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        45
           1       1.00      0.99      0.99        75

    accuracy                           0.99       120
   macro avg       0.99      0.99      0.99       120
weighted avg       0.99      0.99      0.99       120



In [9]:
from sklearn.metrics import roc_auc_score
log_roc=roc_auc_score(y_test,logistic_grid.predict_proba(x_test)[:,1])
log_roc

np.float64(1.0)

In [10]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
param_grid={'kernel':['rbf','linear','poly'],'gamma':['auto','scale']}
svc_grid=GridSearchCV(SVC(probability=True),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
svc_grid.fit(x_train,y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=SVC(probability=True), n_jobs=-1,
             param_grid={'gamma': ['auto', 'scale'],
                         'kernel': ['rbf', 'linear', 'poly']},
             scoring='f1_weighted', verbose=3)

In [11]:
y_pred=svc_grid.predict(x_test)
from sklearn.metrics import confusion_matrix  
cm1=confusion_matrix(y_test,y_pred)
from sklearn.metrics import classification_report
clf_report1=classification_report(y_test,y_pred)
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,y_pred,average='weighted')
print("The f1_macro value for best parameter{}:".format(svc_grid.best_params_),f1_macro)
print("The confusion Matrix is:\n",cm1)
print("The Cassification report is:\n",clf_report1)
from sklearn.metrics import roc_auc_score
svc_roc=roc_auc_score(y_test,svc_grid.predict_proba(x_test)[:,1])
svc_roc

The f1_macro value for best parameter{'gamma': 'auto', 'kernel': 'linear'}: 0.9751481237656352
The confusion Matrix is:
 [[45  0]
 [ 3 72]]
The Cassification report is:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97        45
           1       1.00      0.96      0.98        75

    accuracy                           0.97       120
   macro avg       0.97      0.98      0.97       120
weighted avg       0.98      0.97      0.98       120



np.float64(0.9994074074074074)

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
param_grid={'criterion':['gini','entropy','log_loss'],'splitter':['best','random']}
tree_grid=GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
tree_grid.fit(x_train,y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'splitter': ['best', 'random']},
             scoring='f1_weighted', verbose=3)

In [13]:
grid_pred=tree_grid.predict(x_test)
from sklearn.metrics import confusion_matrix  
cm2=confusion_matrix(y_test,y_pred)
from sklearn.metrics import classification_report
clf_report2=classification_report(y_test,y_pred)
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,y_pred,average='weighted')
print("The f1_macro value for best parameter{}:",format(tree_grid.best_params_),f1_macro)
print("The confusion Matrix is:\n",cm2)
print("The Cassification report is:\n",clf_report2)

The f1_macro value for best parameter{}: {'criterion': 'gini', 'splitter': 'random'} 0.9751481237656352
The confusion Matrix is:
 [[45  0]
 [ 3 72]]
The Cassification report is:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97        45
           1       1.00      0.96      0.98        75

    accuracy                           0.97       120
   macro avg       0.97      0.98      0.97       120
weighted avg       0.98      0.97      0.98       120



In [14]:
from sklearn.metrics import roc_auc_score
tree_roc=roc_auc_score(y_test,tree_grid.predict_proba(x_test)[:,1])
tree_roc

np.float64(0.98)

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
param_grid={'criterion':['gini','entropy','log_loss'],'max_features':['sqrt','log2'],'n_estimators':[100]}
rf_grid=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
rf_grid.fit(x_train,y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [100]},
             scoring='f1_weighted', verbose=3)

In [16]:
grid_pred=rf_grid.predict(x_test)
from sklearn.metrics import confusion_matrix
cm3=confusion_matrix(y_test,grid_pred)
from sklearn.metrics import classification_report
clf_report3=classification_report(y_test,grid_pred)
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_pred,average='weighted')
print("The f1_macro value for best parameter{}:",format(rf_grid.best_params_),f1_macro)
print("The confusion Matrix is:\n",cm3)
print("The Cassification report is:\n",clf_report3)
from sklearn.metrics import roc_auc_score
rf_roc=roc_auc_score(y_test,rf_grid.predict_proba(x_test)[:,1])
rf_roc

The f1_macro value for best parameter{}: {'criterion': 'gini', 'max_features': 'sqrt', 'n_estimators': 100} 0.9833333333333333
The confusion Matrix is:
 [[44  1]
 [ 1 74]]
The Cassification report is:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98        45
           1       0.99      0.99      0.99        75

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120



np.float64(0.9997037037037035)

In [17]:
re=logistic_grid.cv_results_
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003202,0.001662,0.000000,0.000000,l1,lbfgs,"{'penalty': 'l1', 'solver': 'lbfgs'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
1,0.052770,0.020236,0.059744,0.040924,l1,liblinear,"{'penalty': 'l1', 'solver': 'liblinear'}",1.000000,0.982221,0.982221,0.947015,0.963912,0.975074,0.018085,4
2,0.002077,0.000677,0.000000,0.000000,l1,newton_cg,"{'penalty': 'l1', 'solver': 'newton_cg'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
3,0.073841,0.008559,0.023120,0.003901,l1,saga,"{'penalty': 'l1', 'solver': 'saga'}",1.000000,0.982221,0.982221,0.964572,0.963912,0.978585,0.013392,3
4,0.068164,0.026618,0.026427,0.003812,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.982221,1.000000,0.982221,1.000000,1.000000,0.992888,0.008710,1
5,0.013619,0.003297,0.025792,0.004750,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.964572,0.982221,0.964572,0.964572,0.981894,0.971566,0.008567,5
6,0.001812,0.000082,0.000000,0.000000,l2,newton_cg,"{'penalty': 'l2', 'solver': 'newton_cg'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
7,0.046176,0.010210,0.020929,0.003409,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.982221,0.982221,0.964572,0.982221,0.981894,0.978626,0.007028,2


In [18]:
import pickle
filename="CKD_model_creation.sav"

In [19]:
pickle.dump(logistic_grid,open(filename,'wb'))

In [20]:
loaded_model=pickle.load(open("CKD_model_creation.sav",'rb'))
future_prediction=loaded_model.predict([[5.0,76.45,3.0,0.0,152.11,57.58,5.08,202.15,4.70,12.51,38.86,8408.01,5.2,0,0,1,1,1,1,0,0,1,0,1,0,0,1]])
print("Future predictions{}:".format(future_prediction))

Future predictions[0]:
